In [2]:
import pandas as pd
from konlpy.tag import Okt
import numpy as np
import gensim.corpora as corpora
import re
import gensim
from pprint import pprint

In [165]:
df = pd.read_csv('find.csv')
df.head()
len(df)

16708

In [290]:
drop_df = df.drop_duplicates(keep='first')
len(drop_df)

15129

In [291]:
drop_df['href'].fillna(method='ffill',inplace=True)
drop_df['title'].fillna(method='ffill',inplace=True)
drop_df['reviewNum'].fillna(method='ffill',inplace=True)
drop_df['tag'].fillna(method='ffill',inplace=True)
drop_df['brand'].fillna(method='ffill',inplace=True)
drop_df['company'].fillna(method='ffill',inplace=True)
drop_df['howToUse'].fillna(method='ffill',inplace=True)
drop_df['ingredients'].fillna(method='ffill',inplace=True)
drop_df['image'].fillna(method='ffill',inplace=True)
drop_df['volume'].fillna(method='ffill',inplace=True)
drop_df['price'].fillna(method='ffill',inplace=True)
# drop_df = drop_df[drop_df['tag'].apply(lambda x: str(x) == '두피샴푸' or str(x) == '탈모샴푸')]
drop_df



/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/762488147.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  drop_df['href'].fillna(method='ffill',inplace=True)
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/762488147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_df['href'].fillna(method='ffill',inplace=True)
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/762488147.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  drop_df['title'].fillna(method='ffill',inplace=True)
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/762488147.py:2: SettingWithCopyW

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,price,totalScore,satisfactionScore,priceScore,rebuyScore,commenter,commentDate,commentContent,commentGood,commentBad
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",NaN,80%,80%,100%,K2646350517,3달 전,NaN,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ..."
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,a337*****,한 시간 전,탈모라 써봤는데 시원하고 좋아요,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,suwo******,하루 전,머리감을때마다 시원하고좋아요,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,wall***,하루 전,전에 쿨샴푸를 한번썻는데 맘에들어서 다른색으로 하나 더 주문했습니다. 샘플 사은품...,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,zzzz****,2일 전,아주좋습니다좋아요~,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,"36,000원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16704,https://daedamo.com/ingre/82?sca=탈모관련상품&overla...,\n 트리코민 덴시파잉 샴푸,0.0,두피샴푸,트리코민,니옥신,거품을 충분히 내신 후 바로 헹구지 마시고 3-5분 가량 그대로 두어 영양성분이 충...,"정제수, 알로에베라잎즙, 에키네시하추출물, 아이소말트, 완두싹추출물, 하이드롤라이즈...",https://daedamo.com/new/data/file/ingre/179434...,177.4ml,"39,800원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16705,https://daedamo.com/ingre/80?sca=탈모관련상품&overla...,\n 드림헤어 순간증모제 전용 미스트,0.0,스타일링,드림헤어,니옥신,증모제를 사용하신후 본 제품을 직접적으로 분사하지 마시고 머리위 하늘에 뿌려주듯 분...,"에탄올, 정제수, 아크릴레이트/옥틸아크릴아마이드코폴리머, 녹차추출물, 곰솔잎추출물,...",https://daedamo.com/new/data/file/ingre/179434...,150ml,"8,000원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16706,https://daedamo.com/ingre/75?sca=탈모관련상품&overla...,\n 드림헤어 블랙시크릿(휴대용),0.0,헤어커버,드림헤어,니옥신,증모제를 도포후 두피쪽에 증착될 수 있도록 머리를 쓰다듬듯이 살살 털어줍니다.,"레이온, 폴라아마이드, 비오틴, 실크펩타이드, 카퍼트리펩타이드, 대두레시틴, 헤나추출물",https://daedamo.com/new/data/file/ingre/179434...,7g,"15,000원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [169]:
# df = drop_df
# df.nunique()
len(drop_df)

12168

In [170]:
content = drop_df[drop_df['commentContent'].notnull()]['commentContent']
good = drop_df[drop_df['commentGood'].notnull()]['commentGood']
bad = drop_df[drop_df['commentBad'].notnull()]['commentBad']
print(len(content),len(good),len(bad))
content

916 9874 5739


1                                      탈모라 써봤는데 시원하고 좋아요 
2                                        머리감을때마다 시원하고좋아요 
3        전에 쿨샴푸를 한번썻는데 맘에들어서 다른색으로 하나 더 주문했습니다. 샘플 사은품...
4                                        아주좋습니다좋아요~      
5            향이 진하게 나고 시원한 감이 있어서 좋아요 거품 밀도도 좋고요         
                              ...                        
1171                                         감사합니다 번창하세오 
1172     배송이빠르게와서 한번이용해보았습니다괜찮아서 앞으로도 계속구매해볼까생각중입니다      
1173     쿠팡으로 사서 써보고 좋아서 부모님댁에도주문해드렸어요.제가 10년째 탈모샴푸를 쓰...
1174     거품이 신기하고 시원한 탈모샴푸가 궁금해서 구매했는데 와 평소에 없던 비듬이 생겨...
1175     거품도 풍성하게 나고 머릿결도 좋아진 것 같아요꾸준히 쓰면 모발에 힘도 나겠죠? ...
Name: commentContent, Length: 916, dtype: object

In [76]:
tokenizer = Okt()

In [171]:
pattern = re.compile(r'[\n\t]+')
def make_tokens(doc):
    tokens = []
    if doc and (type(doc)==str or not np.isnan(doc)):
        cleaned_doc = pattern.sub(' ',doc).strip()
        phrase = tokenizer.pos(cleaned_doc,norm=True,stem=True)
        tokens = [word[0] for word in phrase if word[0] and (word[1] in ['Noun','Adjective','Verb','Adverb','VerbPrefix','Suffix'] and (word[0] not in ['하다','있다']))]
    return tokens

In [172]:
cont_tokens = []
for t in content:
    cont_tokens.append(make_tokens(t))
cont_tokens

[['탈모', '써다', '보다', '시원하다', '좋다'],
 ['머리', '감', '때', '시원하다', '좋다'],
 ['전',
  '쿨',
  '샴푸',
  '한번',
  '썻',
  '늘다',
  '맘',
  '들어서다',
  '색',
  '하나',
  '더',
  '주문',
  '샘플',
  '사은',
  '품다',
  '정말',
  '좋다',
  '여'],
 ['아주', '좋다', '좋다'],
 ['향', '진하다', '나다', '시원하다', '감', '좋다', '거품', '밀도', '좋다'],
 ['맨솔',
  '약하다',
  '좋다',
  '통',
  '문제',
  '건지다',
  '샴푸',
  '스물',
  '스물',
  '입구',
  '나오다',
  '빼다',
  '좋다'],
 ['테', '안', '맞다'],
 ['강력하다',
  '쿨링',
  '샴푸',
  '원하다',
  '구입',
  '쿨링',
  '감다',
  '강하다',
  '않다',
  '탄산',
  '땜',
  '더',
  '강하다',
  '생각',
  '생각',
  '약하다',
  '정력',
  '좋다',
  '머리',
  '결',
  '부드럽다',
  '맘',
  '드네'],
 ['좋다', '빨르다', '재', '구매', '최고', '감사'],
 ['시원하다', '부드럽다'],
 ['통째', '쓰다', '중', '이다', '크다', '단점', '없다', '샴푸', '무난', '쓰기', '좋다'],
 ['빠르다', '배송', '감사하다'],
 ['탈모', '효과', '같다', '시원하다'],
 ['자다', '쓰다', '통째', '두피', '시원하다'],
 ['샴푸', '비다', '시원하다', '좋다'],
 ['샴푸', '비다', '많이', '시원하다', '좋다', '여'],
 ['멏년',
  '동안',
  '안',
  '써다',
  '보다',
  '삼',
  '푸다',
  '들이다',
  '없다',
  '해외',
  '직구',
  '비싸다',
  '제품',
  '들',
  '들

In [178]:
len(cont_tokens)


916

In [174]:
good_tokens = []
for t in good:
    good_tokens.append(make_tokens(t))
good_tokens

[['앞머리',
  '모발',
  '이식',
  '프로',
  '스카',
  '띄엄띄엄',
  '먹다',
  '정수리',
  '광탈',
  '시작',
  '곧바로',
  '미녹실딜',
  '다모',
  '다트',
  '진행',
  '요',
  '샴푸',
  '쿠팡',
  '행사',
  '때',
  '사악하다',
  '가격',
  '보고',
  '그냥',
  '속',
  '늘다',
  '사다',
  '오다',
  '놓다',
  '거',
  '엘땡',
  '건강',
  '닥터',
  '사기꾼',
  '태평',
  '레',
  '뤼어',
  '따위',
  '어',
  '탈모',
  '이름',
  '꺼내다',
  '안되다',
  '쓰레기',
  '확실하다',
  '두피',
  '열',
  '각질',
  '비듬',
  '개선',
  '과',
  '뚜렸함',
  '알다',
  '샴푸',
  '머리',
  '나다',
  '직접',
  '적',
  '상관없다',
  '두피',
  '열도',
  '사실',
  '기분',
  '탓',
  '밉다',
  '녹다',
  '딜로',
  '확장',
  '되다',
  '내',
  '두피',
  '모낭',
  '쾌적하다',
  '환경',
  '만들다',
  '보다',
  '되다',
  '발포',
  '기술',
  '요',
  '손등',
  '살짝',
  '문지르다',
  '보다',
  '무슨',
  '매직',
  '같다',
  '부글부글',
  '방울',
  '커지다',
  '신기하다',
  '여',
  '틀다',
  '비싸다',
  '딱',
  '번만',
  '짜다',
  '거품',
  '장난',
  '아니다',
  '부글부글',
  '그냥',
  '속',
  '늘다',
  '한번',
  '씩',
  '들다',
  '써다',
  '보삼',
  '나',
  '만족하다',
  '좀',
  '더',
  '할인',
  '해주다',
  '삼',
  '그',
  '스댕',
  '용기',
  '리',
  '필로',
  '만들다',
  '가격',
  

In [175]:
len(good_tokens)

9874

In [176]:
bad_tokens = []
for t in bad:
    bad_tokens.append(make_tokens(t))
bad_tokens

[['스댕',
  '용기',
  '개',
  '이쁘다',
  '좋다',
  '원가',
  '너무',
  '비싸다',
  '내',
  '보기',
  '가격',
  '높다',
  '최대',
  '원인',
  '듯함',
  '요',
  '튜브',
  '만들다',
  '삼',
  '좀',
  '흐르다',
  '굳다',
  '요',
  '피',
  '같다',
  '돈',
  '아깝다',
  '움',
  '근데',
  '장점',
  '다',
  '커버',
  '함'],
 ['음',
  '꼭',
  '찾다',
  '적다',
  '가격',
  '탈모',
  '샴푸',
  '들',
  '다',
  '들다',
  '비싸다',
  '근데',
  '뭐',
  '다비',
  '싸다',
  '요'],
 ['스댕',
  '용기',
  '개',
  '이쁘다',
  '좋다',
  '원가',
  '너무',
  '비싸다',
  '내',
  '보기',
  '가격',
  '높다',
  '최대',
  '원인',
  '듯함',
  '요',
  '튜브',
  '만들다',
  '삼',
  '좀',
  '흐르다',
  '굳다',
  '요',
  '피',
  '같다',
  '돈',
  '아깝다',
  '움',
  '근데',
  '장점',
  '다',
  '커버',
  '함'],
 ['가격',
  '조금',
  '비싸다',
  '그래도',
  '새롭다',
  '기술',
  '적용',
  '되어다',
  '엄청나다',
  '쿨링감',
  '가지',
  '금액',
  '조금',
  '나가다',
  '수',
  '생각',
  '나쁘다',
  '점',
  '아니다',
  '사용',
  '상',
  '문제',
  '되다',
  '없다',
  '탄산',
  '쿨링',
  '샴푸',
  '사용',
  '때',
  '펌핑',
  '입구',
  '부분',
  '탄산',
  '거품',
  '남다',
  '인지',
  '사용',
  '되다',
  '듯'],
 ['특별하다', '없다'],
 ['딱하다', '없다'],
 ['샴푸',

In [177]:
len(bad_tokens)

5739

In [179]:
cont_id2word = corpora.Dictionary(cont_tokens)
cont_corpus = [cont_id2word.doc2bow(text) for text in cont_tokens]

In [180]:
good_id2word = corpora.Dictionary(good_tokens)
good_corpus = [good_id2word.doc2bow(text) for text in good_tokens]

In [181]:
bad_id2word = corpora.Dictionary(bad_tokens)
bad_corpus = [bad_id2word.doc2bow(text) for text in bad_tokens]

In [101]:
import pyLDAvis.gensim
import pickle
import pyLDAvis
import os

In [182]:
num_topics = 5
lda_model5 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model5[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model5,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [122]:
num_topics = 6
lda_model6 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model6[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model6,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [190]:
num_topics = 7
lda_model7 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model7[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model7,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [227]:
def make_topictable(ldamodel,corpus):
    rows = []
    # topic_table = pd.DataFrame()
    for i, topic_list in enumerate(ldamodel[corpus]):
        doc = topic_list[0] if ldamodel.per_word_topics else topic_list
        doc = sorted(doc, key=lambda x: (x[1]), reverse = True)
        # for j, (topic_num, prop_topic) in enumerate(doc):
        #     if j == 0:
        #         topic_table = pd.concat([topic_table,pd.Series([int(topic_num),round(prop_topic,4),topic_list])],ignore_index=True)
        #     else:
        #         break
        for j, (topic_num, prop_topic) in enumerate(doc):
            if j == 0:
                row = [int(topic_num), round(prop_topic, 4), topic_list]
                rows.append(row)
    
    topic_table = pd.concat([pd.Series(row) for row in rows], axis=1).T
    # topic_table.columns = ['Topic_Num', 'Prop_Topic', 'Topic_List']
    return(topic_table)

In [191]:
cont_lda_model = lda_model7

In [229]:
cont_topictable = make_topictable(cont_lda_model,cont_corpus)
cont_topictable
cont_topictable.reset_index()
cont_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
cont_topictable
# topictable = topictable.reset_index()
# topictable.columns = 

,문서 번호,가장 비중이 높은 토픽의 비중,각 토픽의 비중
0,4,0.8566,"[(0, 0.023957323), (1, 0.02386534), (2, 0.0239..."
1,5,0.8567,"[(0, 0.023860294), (1, 0.023860116), (2, 0.023..."
2,2,0.9547,"[(2, 0.954702)]"
3,1,0.7851,"[(0, 0.035773713), (1, 0.7850679), (2, 0.03578..."
4,4,0.914,"[(0, 0.014322489), (1, 0.014334685), (2, 0.014..."
...,...,...,...
911,4,0.714,"[(0, 0.04765368), (1, 0.047642414), (2, 0.0476..."
912,3,0.9427,"[(3, 0.94270885)]"
913,2,0.9944,"[(2, 0.9944483)]"
914,3,0.9761,"[(3, 0.9760916)]"


In [185]:
num_topics = 8
lda_model8 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model8[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model8,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [119]:
num_topics = 9
lda_model9 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model9[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model9,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [106]:
num_topics = 10
lda_model10 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model10[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model10,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [194]:
num_topics = 5
good_model = gensim.models.LdaMulticore(corpus=good_corpus,id2word=good_id2word,num_topics=num_topics,iterations=500)
doc_lda = good_model[good_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(good_model,good_corpus,good_id2word)
pyLDAvis.display(vis)

In [224]:
num_topics = 6
good_model2 = gensim.models.LdaMulticore(corpus=good_corpus,id2word=good_id2word,num_topics=num_topics,iterations=500)
doc_lda = good_model2[good_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(good_model2,good_corpus,good_id2word)
pyLDAvis.display(vis)

In [228]:
good_topictable = make_topictable(good_model2,good_corpus)
good_topictable.reset_index()
good_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
good_topictable
# topictable = topictable.reset_index()
# topictable.columns = 

,문서 번호,가장 비중이 높은 토픽의 비중,각 토픽의 비중
0,0,0.9935,"[(0, 0.99347997)]"
1,3,0.9847,"[(3, 0.9846592)]"
2,5,0.5424,"[(3, 0.42953652), (5, 0.5423631)]"
3,2,0.9852,"[(2, 0.9851987)]"
4,4,0.9844,"[(4, 0.9843982)]"
...,...,...,...
9869,3,0.8946,"[(0, 0.021041287), (1, 0.021071786), (2, 0.021..."
9870,2,0.8795,"[(0, 0.024082024), (1, 0.024048617), (2, 0.879..."
9871,4,0.831,"[(0, 0.033727035), (1, 0.033743743), (2, 0.033..."
9872,2,0.8314,"[(0, 0.033679746), (1, 0.03383373), (2, 0.8314..."


In [232]:
num_topics = 7
bad_model = gensim.models.LdaMulticore(corpus=bad_corpus,id2word=bad_id2word,num_topics=num_topics,iterations=500)
doc_lda = bad_model[bad_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(bad_model,bad_corpus,bad_id2word)
pyLDAvis.display(vis)

In [246]:
num_topics = 9
bad_model2 = gensim.models.LdaMulticore(corpus=bad_corpus,id2word=bad_id2word,num_topics=num_topics,iterations=500)
doc_lda = bad_model2[bad_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(bad_model2,bad_corpus,bad_id2word)
pyLDAvis.display(vis)

In [248]:
bad_topictable = make_topictable(bad_model2,bad_corpus)
bad_topictable.reset_index()
bad_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
bad_topictable
# topictable = topictable.reset_index()
# topictable.columns = 

,문서 번호,가장 비중이 높은 토픽의 비중,각 토픽의 비중
0,1,0.9738,"[(1, 0.97384095)]"
1,5,0.7119,"[(5, 0.71185887), (7, 0.24233761)]"
2,1,0.9738,"[(1, 0.97384083)]"
3,7,0.796,"[(0, 0.18455222), (7, 0.7959838)]"
4,4,0.7035,"[(0, 0.037062634), (1, 0.037055537), (2, 0.037..."
...,...,...,...
5734,2,0.8729,"[(0, 0.015879462), (1, 0.015887208), (2, 0.872..."
5735,0,0.8729,"[(0, 0.87293535), (1, 0.015883282), (2, 0.0158..."
5736,4,0.9012,"[(0, 0.012351962), (1, 0.012351288), (2, 0.012..."
5737,4,0.7777,"[(0, 0.027782874), (1, 0.027783778), (2, 0.027..."


In [260]:
drop_df

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16704,https://daedamo.com/ingre/82?sca=탈모관련상품&overla...,\n 트리코민 덴시파잉 샴푸,0.0,두피샴푸,트리코민,니옥신,거품을 충분히 내신 후 바로 헹구지 마시고 3-5분 가량 그대로 두어 영양성분이 충...,"정제수, 알로에베라잎즙, 에키네시하추출물, 아이소말트, 완두싹추출물, 하이드롤라이즈...",https://daedamo.com/new/data/file/ingre/179434...,177.4ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16705,https://daedamo.com/ingre/80?sca=탈모관련상품&overla...,\n 드림헤어 순간증모제 전용 미스트,0.0,스타일링,드림헤어,니옥신,증모제를 사용하신후 본 제품을 직접적으로 분사하지 마시고 머리위 하늘에 뿌려주듯 분...,"에탄올, 정제수, 아크릴레이트/옥틸아크릴아마이드코폴리머, 녹차추출물, 곰솔잎추출물,...",https://daedamo.com/new/data/file/ingre/179434...,150ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16706,https://daedamo.com/ingre/75?sca=탈모관련상품&overla...,\n 드림헤어 블랙시크릿(휴대용),0.0,헤어커버,드림헤어,니옥신,증모제를 도포후 두피쪽에 증착될 수 있도록 머리를 쓰다듬듯이 살살 털어줍니다.,"레이온, 폴라아마이드, 비오틴, 실크펩타이드, 카퍼트리펩타이드, 대두레시틴, 헤나추출물",https://daedamo.com/new/data/file/ingre/179434...,7g,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [267]:
drop_df['contentTopicPerc'] = np.nan
drop_df['contentTopicDist'] = np.nan
drop_df['goodTopic'] = np.nan
drop_df['goodTopicPerc'] = np.nan
drop_df['goodTopicDist'] = np.nan
drop_df['badTopic'] = np.nan
drop_df['badTopicPerc'] = np.nan
drop_df['badTopicDist'] = np.nan

# drop_df = drop_df.drop(columns='contentTopic',axis=1)
drop_df.drop(columns = ['href','title','reviewNum','tag','brand','company','ingredients','image','volume','price','totalScore'])

,howToUse,satisfactionScore,priceScore,rebuyScore,commenter,commentDate,commentContent,commentGood,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,80%,80%,100%,K2646350517,3달 전,NaN,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,NaN,NaN,NaN,a337*****,한 시간 전,탈모라 써봤는데 시원하고 좋아요,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,NaN,NaN,NaN,suwo******,하루 전,머리감을때마다 시원하고좋아요,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,NaN,NaN,NaN,wall***,하루 전,전에 쿨샴푸를 한번썻는데 맘에들어서 다른색으로 하나 더 주문했습니다. 샘플 사은품...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,NaN,NaN,NaN,zzzz****,2일 전,아주좋습니다좋아요~,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16703,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16704,거품을 충분히 내신 후 바로 헹구지 마시고 3-5분 가량 그대로 두어 영양성분이 충...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16705,증모제를 사용하신후 본 제품을 직접적으로 분사하지 마시고 머리위 하늘에 뿌려주듯 분...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16706,증모제를 도포후 두피쪽에 증착될 수 있도록 머리를 쓰다듬듯이 살살 털어줍니다.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [273]:
contentTopic = cont_topictable['문서 번호']
contentTopicPerc = cont_topictable['가장 비중이 높은 토픽의 비중']
contentTopicDist = cont_topictable['각 토픽의 비중']

goodTopic = good_topictable['문서 번호']
goodTopicPerc = good_topictable['가장 비중이 높은 토픽의 비중']
goodTopicDist = good_topictable['각 토픽의 비중']

badTopic = bad_topictable['문서 번호']
badTopicPerc = bad_topictable['가장 비중이 높은 토픽의 비중']
badTopicDist = bad_topictable['각 토픽의 비중']

916


In [289]:
contentTopicDist[0]

[(0, 0.023957323),
 (1, 0.02386534),
 (2, 0.023919415),
 (3, 0.023868967),
 (4, 0.856599),
 (5, 0.023906127),
 (6, 0.023883833)]

In [342]:
cont_hrefs = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['href'].reset_index()
cont_commenter = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['commenter'].reset_index()
cont_comment = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['commentContent'].reset_index()

good_hrefs = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentGood'].notnull()]['href'].reset_index()
good_commenter = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentGood'].notnull()]['commenter'].reset_index()
good_comment = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentGood'].notnull()]['commentGood'].reset_index()

bad_hrefs = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentBad'].notnull()]['href'].reset_index()
bad_commenter = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentBad'].notnull()]['commenter'].reset_index()
bad_comment = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentBad'].notnull()]['commentBad'].reset_index()

# cont_hrefs = cont_hrefs.reset_index()
cont_hrefs
cont_commenter
# cont_comment

/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/3153998612.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cont_hrefs = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['href'].reset_index()
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/3153998612.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cont_commenter = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['commenter'].reset_index()
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/3153998612.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  cont_comment = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['commentContent'].reset_index()
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39

,index,commenter
0,1,a337*****
1,2,suwo******
2,3,wall***
3,4,zzzz****
4,5,sj05****
...,...,...
911,1171,bbho****
912,1172,ggod****
913,1173,love*******
914,1174,the7****


In [316]:
cont_comment.index

RangeIndex(start=0, stop=916, step=1)

In [338]:
drop_df['goodTopic'] = None
drop_df['goodTopicPerc'] = None
drop_df['goodTopicDist'] = None
drop_df['badTopic'] = None
drop_df['badTopicPerc'] = None
drop_df['badTopicDist'] = None


/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/655073321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_df['goodTopic'] = None
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/655073321.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_df['goodTopicPerc'] = None
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/655073321.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

In [339]:
drop_df

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None,None,None,None,None,None,None
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,4,0.8566,"[(0, 0.023957323), (1, 0.02386534), (2, 0.0239...",None,None,None,None,None,None
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,5,0.8567,"[(0, 0.023860294), (1, 0.023860116), (2, 0.023...",None,None,None,None,None,None
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2,0.9547,"[(2, 0.954702)]",None,None,None,None,None,None
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,1,0.7851,"[(0, 0.035773713), (1, 0.7850679), (2, 0.03578...",None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16704,https://daedamo.com/ingre/82?sca=탈모관련상품&overla...,\n 트리코민 덴시파잉 샴푸,0.0,두피샴푸,트리코민,니옥신,거품을 충분히 내신 후 바로 헹구지 마시고 3-5분 가량 그대로 두어 영양성분이 충...,"정제수, 알로에베라잎즙, 에키네시하추출물, 아이소말트, 완두싹추출물, 하이드롤라이즈...",https://daedamo.com/new/data/file/ingre/179434...,177.4ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16705,https://daedamo.com/ingre/80?sca=탈모관련상품&overla...,\n 드림헤어 순간증모제 전용 미스트,0.0,스타일링,드림헤어,니옥신,증모제를 사용하신후 본 제품을 직접적으로 분사하지 마시고 머리위 하늘에 뿌려주듯 분...,"에탄올, 정제수, 아크릴레이트/옥틸아크릴아마이드코폴리머, 녹차추출물, 곰솔잎추출물,...",https://daedamo.com/new/data/file/ingre/179434...,150ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16706,https://daedamo.com/ingre/75?sca=탈모관련상품&overla...,\n 드림헤어 블랙시크릿(휴대용),0.0,헤어커버,드림헤어,니옥신,증모제를 도포후 두피쪽에 증착될 수 있도록 머리를 쓰다듬듯이 살살 털어줍니다.,"레이온, 폴라아마이드, 비오틴, 실크펩타이드, 카퍼트리펩타이드, 대두레시틴, 헤나추출물",https://daedamo.com/new/data/file/ingre/179434...,7g,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [327]:
cont_hrefs

,index,href
0,1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
1,2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
2,3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
3,4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
4,5,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
...,...,...
911,1171,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...
912,1172,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...
913,1173,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...
914,1174,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...


In [337]:
for i in cont_hrefs.index:
    condition = (drop_df['href'] == cont_hrefs['href'][i]) & (drop_df['commenter'] == cont_commenter['commenter'][i]) & (drop_df['commentContent'] == cont_comment['commentContent'][i])
    drop_df.loc[condition,'contentTopic'] = contentTopic[i]
    drop_df.loc[condition,'contentTopicPerc'] = contentTopicPerc[i]
    drop_df.loc[condition,'contentTopicDist'] = str(contentTopicDist[i])
drop_df.head()

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,priceScore,rebuyScore,commenter,commentDate,commentContent,commentGood,commentBad,contentTopic,contentTopicPerc,contentTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,80%,100%,K2646350517,3달 전,NaN,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,a337*****,한 시간 전,탈모라 써봤는데 시원하고 좋아요,NaN,NaN,4,0.8566,"[(0, 0.023957323), (1, 0.02386534), (2, 0.0239..."
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,suwo******,하루 전,머리감을때마다 시원하고좋아요,NaN,NaN,5,0.8567,"[(0, 0.023860294), (1, 0.023860116), (2, 0.023..."
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,wall***,하루 전,전에 쿨샴푸를 한번썻는데 맘에들어서 다른색으로 하나 더 주문했습니다. 샘플 사은품...,NaN,NaN,2,0.9547,"[(2, 0.954702)]"
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,zzzz****,2일 전,아주좋습니다좋아요~,NaN,NaN,1,0.7851,"[(0, 0.035773713), (1, 0.7850679), (2, 0.03578..."


In [343]:
for i in good_hrefs.index:
    condition = (drop_df['href'] == good_hrefs['href'][i]) & (drop_df['commenter'] == good_commenter['commenter'][i]) & (drop_df['commentGood'] == good_comment['commentGood'][i])
    drop_df.loc[condition,'goodTopic'] = goodTopic[i]
    drop_df.loc[condition,'goodTopicPerc'] = goodTopicPerc[i]
    drop_df.loc[condition,'goodTopicDist'] = str(goodTopicDist[i])
drop_df.head()

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None,0,0.9935,"[(0, 0.9934807)]",None,None,None
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,4,0.8566,"[(0, 0.023957323), (1, 0.02386534), (2, 0.0239...",None,None,None,None,None,None
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,5,0.8567,"[(0, 0.023860294), (1, 0.023860116), (2, 0.023...",None,None,None,None,None,None
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2,0.9547,"[(2, 0.954702)]",None,None,None,None,None,None
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,1,0.7851,"[(0, 0.035773713), (1, 0.7850679), (2, 0.03578...",None,None,None,None,None,None


In [344]:
for i in bad_hrefs.index:
    condition = (drop_df['href'] == bad_hrefs['href'][i]) & (drop_df['commenter'] == bad_commenter['commenter'][i]) & (drop_df['commentBad'] == bad_comment['commentBad'][i])
    drop_df.loc[condition,'badTopic'] = badTopic[i]
    drop_df.loc[condition,'badTopicPerc'] = badTopicPerc[i]
    drop_df.loc[condition,'badTopicDist'] = str(badTopicDist[i])
drop_df.head()

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None,0,0.9935,"[(0, 0.9934807)]",1,0.9738,"[(1, 0.97384083)]"
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,4,0.8566,"[(0, 0.023957323), (1, 0.02386534), (2, 0.0239...",None,None,None,None,None,None
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,5,0.8567,"[(0, 0.023860294), (1, 0.023860116), (2, 0.023...",None,None,None,None,None,None
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2,0.9547,"[(2, 0.954702)]",None,None,None,None,None,None
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,1,0.7851,"[(0, 0.035773713), (1, 0.7850679), (2, 0.03578...",None,None,None,None,None,None


In [348]:
drop_df[drop_df['badTopic'].notnull()]

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None,0,0.9935,"[(0, 0.9934807)]",1,0.9738,"[(1, 0.97384083)]"
501,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"음 꼭찾아서 적으라면,, 가격? 탈모샴푸들 다들 비싸서 근데 뭐~ 다비싸니까요~ ...",None,None,None,4,0.987,"[(4, 0.987028)]",5,0.7119,"[(5, 0.71185887), (7, 0.24233761)]"
542,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None,0,0.9935,"[(0, 0.9934807)]",1,0.9738,"[(1, 0.97384083)]"
573,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,가격이 조금 비싸긴 합니다. 그래도 새로운 기술이 적용되어 엄청난 쿨링감을 가지고...,None,None,None,4,0.9859,"[(4, 0.98594284)]",7,0.796,"[(0, 0.18455222), (7, 0.7959838)]"
781,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,특별히 없어요,None,None,None,5,0.9835,"[(5, 0.9834675)]",4,0.7035,"[(0, 0.037062634), (1, 0.037055537), (2, 0.037..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14073,https://daedamo.com/ingre/678?sca=탈모관련상품&overl...,\n 포미포미 맥주 샴푸,1.0,탈모샴푸,포미포미,(주)서울화장품,미온수로 모발 및 두피를 충분히 적신 후 적당량을 모발과 두피에 골고루 도포 후 2...,"정제수, 데실글루코사이드, 디소듐라우레스설포석시네이트, 포타슘코코일글리시네이트, 글...",https://daedamo.com/new/data/file/ingre/179434...,520ml,...,머리가 더빠짐 역시 약이 답이다,None,None,None,3,0.8946,"[(0, 0.021041287), (1, 0.021071786), (2, 0.021...",2,0.8729,"[(0, 0.015879462), (1, 0.015887208), (2, 0.872..."
14074,https://daedamo.com/ingre/2853?sca=탈모관련상품&over...,\n 상모단 샴푸,1.0,두피샴푸,어니스트쥬디,(주)서울화장품,물기가 있는 머리카락에 소량 펌핑하여 샴푸잉.,"인삼수, 흑미추출물, 검정콩추출물, 검은깨추출물, 소듐코코일글루타메이트, 데실글루코...",https://daedamo.com/new/data/file/ingre/373194...,250ml,...,한방샴푸라 그런지 한약재 냄새가 난다,None,None,None,2,0.8795,"[(0, 0.024082024), (1, 0.024048617), (2, 0.879...",0,0.8729,"[(0, 0.87293535), (1, 0.015883282), (2, 0.0158..."
14080,https://daedamo.com/ingre/598?sca=탈모관련상품&overl...,\n 하수오 허벌 에센셜 샴푸,1.0,탈모샴푸,피엘하수오,(주)서울화장품,모발이 젖은 상태에서 적당량을 덜어냅니다. 가볍게 마사지 하듯이 거품을 내어 감습니...,"암모늄라우레스설페이트, 편백수, 암모늄라우릴설페이트, 정제수, 메칠폴리실톡산에멀젼,...",https://daedamo.com/new/data/file/ingre/179434...,750ml,...,구하기 어려운 점? 하지만 도움은 안되는 것 같은 점..?,None,None,None,4,0.831,"[(0, 0.033727035), (1, 0.033743743), (2, 0.033...",4,0.9012,"[(0, 0.012351962), (1, 0.012351288), (2, 0.012..."
14081,https://daedamo.com/ingre/1281?sca=탈모관련상품&over...,\n 헤어캅 네츄럴 샴푸,2.0,두피샴푸,헤어캅,(주)서울화장품,미온수로 두피를 충분히 불려주세요. 샴푸 양은 500원 동전크기만큼으로 거품을 내어...,"어성초, 하수오, 고삼, 녹차, 오미자 등",https://daedamo.com/new/data/file/ingre/179434...,500g,...,아직까진 잘모르겠네요,None,None,None,2,0.8314,"[(0, 0.033679746), (1, 0.03383373), (2, 0.8314...",4,0.7777,"[(0, 0.027782874), (1, 0.027783778), (2, 0.027..."


In [349]:
drop_df.to_csv('final_lda.csv',index=False,encoding='utf-8-sig',escapechar='\\')

In [304]:
drop_df.head()

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,priceScore,rebuyScore,commenter,commentDate,commentContent,commentGood,commentBad,contentTopic,contentTopicPerc,contentTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,80%,100%,K2646350517,3달 전,NaN,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,a337*****,한 시간 전,탈모라 써봤는데 시원하고 좋아요,NaN,NaN,5,0.8567,None
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,suwo******,하루 전,머리감을때마다 시원하고좋아요,NaN,NaN,None,None,None
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,wall***,하루 전,전에 쿨샴푸를 한번썻는데 맘에들어서 다른색으로 하나 더 주문했습니다. 샘플 사은품...,NaN,NaN,None,None,None
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,zzzz****,2일 전,아주좋습니다좋아요~,NaN,NaN,None,None,None


good_topictable = make_topictable(good_model2,good_corpus)
# good_topictable
good_topictable.reset_index()
good_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
good_topictable
# topictable = topictable.reset_index()
# topictable.columns = 